# 03 · Add a small adapter
Keep the base weights fixed. Add a small set of weights that can learn the task.

**Frozen base + trainable LoRA adapter → adapted model**

| Component | Job in this lesson |
|---|---|
| **OpenClaw + lightweight Nano** | Read the skill and coordinate the model-building tools. |
| **Python + PEFT** | Create a tiny model, attach LoRA, and inspect the weights. |
| **Lightning** | Remains the workshop’s serving model. No Lightning inference or training is needed in this CPU exercise. |

This is a **tiny, randomly initialized practice model**, not a smaller pretrained Lightning. It makes the mechanics quick to inspect. We attach the adapter here; training comes in notebook 04.

**CPU training is for demonstration purposes only.** This tiny random model makes LoRA mechanics easy to inspect. For actual NVIDIA Nemotron Lightning GPU training, follow [train_lightning.py](scripts/train_lightning.py) and the optional GPU walkthrough in [notebook 04](04-train-adapter.ipynb). Its adapter cannot be attached to Lightning.


###  parameter-efficient fine-tuning

[PEFT quicktour](https://huggingface.co/docs/peft/main/en/quicktour) for configuring and attaching adapters; [Transformers](https://huggingface.co/docs/transformers/index) for model construction. Our environment pins PEFT 0.21.2 and Transformers 5.19.0; upstream `main` pages can differ.

build the CPU base, attach rank 8, inspect the trainable parameter names, then compare ranks 4/8/16. Saving here writes the base and adapter recipe; training happens in 04. No model service or weight download is needed for the manual exercise.

The executable cells directly construct `GPT2LMHeadModel`, `LoraConfig`, and `get_peft_model`, inspect parameters, and save the base and adapter configuration. The agent recipe remains separate.

the base is randomly initialized and tiny. This demonstrates adapter mechanics, not useful pretrained language knowledge. The adapter cannot be attached to Lightning or a hosted OpenAI model.

**NVIDIA model boundary:** compare the [Lightning model card](https://build.nvidia.com/nvidia/nemotron-3.5-lightning-30b-a3b/modelcard) with this randomly initialized GPT-2 exercise. The PEFT layer targets shown here belong to the practice model; they are not a Lightning adaptation recipe.


**Manual experiment versus agent reference:** the executable cells below are yours to edit. The OpenClaw section at the end runs the fixed workshop recipe in `agent/lessons/`; it does not ingest your edited cells or reuse manual artifacts. Setup, displays, artifact bookkeeping, and terminal/process transport remain helpers.

### Start here
Import Transformers and PEFT. Everything in this lesson is created locally on the CPU; no model download is needed.

In [1]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from transformers import GPT2Config, GPT2LMHeadModel
from peft import LoraConfig, get_peft_model
from course.agent.lesson_walkthrough import table
torch.set_num_threads(2)
TARGETS = ['c_attn', 'c_proj', 'c_fc']
# Change width/layers here; this is a random practice model, not pretrained Lightning.
base_config = GPT2Config(vocab_size=258, n_positions=768, n_embd=48, n_layer=1, n_head=2,
                        bos_token_id=1, eos_token_id=1, pad_token_id=0)
def make_base():
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(17)
        return GPT2LMHeadModel(base_config)
BUILD = ROOT / 'course/outputs/preview03/cpu'
BUILD.mkdir(parents=True, exist_ok=True)

/home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1 · Create the practice model
A model is a collection of layers and their weights. This one has **1 transformer block, 2 attention heads, and a width of 48**.

Run the cell to see its actual configuration.

In [2]:
base = make_base()
base.save_pretrained(BUILD / "base")
frozen_before = {name: value.clone() for name, value in base.state_dict().items()}
print(type(base).__name__)
display(JSON({
    "layers": base.config.n_layer,
    "attention_heads": base.config.n_head,
    "hidden_width": base.config.n_embd,
    "parameters_before_adapter": sum(p.numel() for p in base.parameters()),
}, expanded=True))

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 776.15it/s]

GPT2LMHeadModel


<IPython.core.display.JSON object>

### 2 · Attach LoRA
Instead of updating a whole weight matrix, LoRA learns two smaller matrices, **A** and **B**.

**Layer output = frozen base output + small adapter update**

`rank` controls the size of those smaller matrices. We start at **8**. PEFT freezes the original weights and creates the trainable adapter weights.

Rerunning the next cell creates a fresh base first, so adapters do not stack up.

In [3]:
rank = 8
adapter_config = LoraConfig(r=rank, lora_alpha=2*rank, lora_dropout=0.0, bias='none',
                            task_type='CAUSAL_LM', target_modules=TARGETS, fan_in_fan_out=True)
model = get_peft_model(base, adapter_config)
trainable = [(name, p) for name, p in model.named_parameters() if p.requires_grad]
assert trainable and all('lora_' in name for name, p in trainable)
report = {'only_lora_trainable': True,
          'adapter_matrices': [{'name': name, 'shape': list(p.shape), 'parameters': p.numel()}
                               for name, p in trainable]}
model.print_trainable_parameters()

trainable params: 6,144 || all params: 83,760 || trainable%: 7.3352


### 3 · Inspect the weights that can learn
Run the cell to inspect the actual adapter matrix names and shapes. Names containing `lora_A` and `lora_B` belong to the adapter.

The check must say **only LoRA trainable: True**. These percentages describe this tiny practice model—not the production Lightning adapter.

In [4]:
print("Only LoRA trainable:", report["only_lora_trainable"])
display(JSON(report["adapter_matrices"], expanded=True))

Only LoRA trainable: True


<IPython.core.display.JSON object>

### 4 · Make rank visible
Compare ranks **4, 8, and 16**. Each row builds an independent adapter on the same practice architecture.

Higher rank adds more trainable parameters. It is a capacity choice, not a guarantee of better accuracy. This comparison does not change the adapter you created above.

In [5]:
rank_comparison = []
for candidate_rank in [4, 8, 16]:
    config = LoraConfig(r=candidate_rank, lora_alpha=2*candidate_rank, target_modules=TARGETS,
                        task_type='CAUSAL_LM', fan_in_fan_out=True)
    candidate = get_peft_model(make_base(), config)
    rank_comparison.append((candidate_rank, sum(p.numel() for p in candidate.parameters() if p.requires_grad)))
table(['Rank', 'Trainable parameters'], rank_comparison)

Rank,Trainable parameters
4,3072
8,6144
16,12288


### 5 · Save the starting point
Save two things for the next step:

| Artifact | What it contains |
|---|---|
| `base/` | The original, unadapted practice-model weights. |
| `adapter-config/adapter_config.json` | The recipe for creating the adapter: rank, scale, and target layers. |

These are **not trained adapter weights yet**. The visible check compares every base tensor with its initial value; the unadapted base was saved before LoRA attachment.

In [6]:
for name, value in model.get_base_model().state_dict().items():
    if 'lora_' not in name:
        assert torch.equal(value, frozen_before[name.replace('.base_layer.', '.')]), name
adapter_config.save_pretrained(BUILD / 'adapter-config')
saved = {'base': str(BUILD / 'base'), 'adapter_config': str(BUILD / 'adapter-config/adapter_config.json'),
         'base_weights_unchanged': True, 'training_performed': False}
display(JSON(saved, expanded=True))

<IPython.core.display.JSON object>

In [7]:
adapter_json = json.loads(Path(saved["adapter_config"]).read_text())
display(JSON(adapter_json, expanded=True))

<IPython.core.display.JSON object>

**Read the recipe:** `r` is rank, `lora_alpha` sets the update scale, and `target_modules` selects which layer types receive adapters. Here the scale is `lora_alpha / r = 2`.

These target names belong to the practice GPT-2 architecture. Lightning uses different layer names.

### 6 · Grow the skill library
| Notebook | Skill | What it adds |
|---|---|---|
| 01 | `workshop-model-inventory` | Discover models and test responses. |
| 02 | `workshop-training-data` | Build and check labeled examples. |
| 03 | `workshop-build-lora` | Attach an adapter and check which weights can learn. |

The fixed reference skill gives OpenClaw three actions:

**`inspect-model` → `attach-lora` → `check-lora`**

Nano chooses the calls. The fixed Python tools build and inspect the practice model on the host CPU. They do not train or restart Lightning.

In [8]:
from html import escape
from IPython.display import HTML
skill = (ROOT / "course/agent/skills/workshop-build-lora/SKILL.md").read_text()
display(HTML("<details><summary><b>Read SKILL.md</b></summary><pre style='white-space:pre-wrap'>"
             + escape(skill) + "</pre></details>"))

### 7 · Prepare the terminal demo
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-build-lora/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-build-lora/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

This lesson does not require artifacts from an earlier OpenClaw demo.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [9]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('03-build-lora')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-build-lora/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-model', 'attach-lora', 'check-lora']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

Local run directory: /home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/outputs/agent-demos/03-build-lora/openclaw-3840d00639bf48658d1fd230fe1846c9
OpenClaw sandbox workspace: /home/sandbox/course-runs/openclaw-3840d00639bf48658d1fd230fe1846c9/workspace
Installed skill: /home/sandbox/course-runs/openclaw-3840d00639bf48658d1fd230fe1846c9/workspace/skills/workshop-build-lora/SKILL.md
Python helper: /home/sandbox/course-runs/openclaw-3840d00639bf48658d1fd230fe1846c9/chapter_tool.py

The agent will run these commands individually:
python3 /home/sandbox/course-runs/openclaw-3840d00639bf48658d1fd230fe1846c9/chapter_tool.py inspect-model
python3 /home/sandbox/course-runs/openclaw-3840d00639bf48658d1fd230fe1846c9/chapter_tool.py attach-lora
python3 /home/sandbox/course-runs/openclaw-3840d00639bf48658d1fd230fe1846c9/chapter_tool.py check-lora


### 8 · Open OpenClaw and paste the task
Exit the previous TUI with `/exit` first. The next cell opens a terminal on the workshop computer’s desktop.

The model label says **Nano**, the lightweight orchestration model. Copy only the generated task below into the TUI. Watch it read the skill and call the three tools.

In [10]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

Requested an OpenClaw terminal on the workshop computer’s desktop. When its TUI is ready, paste the task below.
If no window appears, in a Jupyter terminal run: bash /home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/outputs/agent-demos/03-build-lora/openclaw-3840d00639bf48658d1fd230fe1846c9/open-terminal.sh

PASTE THIS INTO OPENCLAW:

Use the workshop-build-lora skill. First read /home/sandbox/course-runs/openclaw-3840d00639bf48658d1fd230fe1846c9/workspace/skills/workshop-build-lora/SKILL.md. The tool path is /home/sandbox/course-runs/openclaw-3840d00639bf48658d1fd230fe1846c9/chapter_tool.py. Task: Create a small CPU model, attach a new LoRA adapter, and check which parameters can change. This is a tiny practice model, not the running Lightning model. Run inspect-model, then attach-lora, then check-lora individually with exec elevated=false, yieldMs=10000, timeoutSeconds=180. If a command is still running, poll with process before continuing. A launcher

### 9 · Bring the results back
Wait until the TUI finishes, then run this cell. Look for **PASS**, **rank 8**, the frozen and trainable parameter counts, and **only_lora_trainable: true**.

This checks the agent’s own artifacts and verifies that it read the skill before executing tools. Running all cells without completing the TUI task will stop here; rerun this cell afterward.

In [11]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

CHECK: PASS
{
  "profile": "cpu",
  "rank": 8,
  "trainable_parameters": 6144,
  "frozen_parameters": 77616,
  "only_lora_trainable": true
}
Saved: /home/nico/dev/nato-workshop/nato-workshop/.workshop/participant-repository/course/outputs/agent-demos/03-build-lora/openclaw-3840d00639bf48658d1fd230fe1846c9


<IPython.core.display.JSON object>

**What you demonstrated:** create a base → attach LoRA → see which weights can learn → compare rank → let Nano coordinate the same work.

**Next:** notebook 04 trains the adapter using the examples from notebook 02. Exit the TUI with `/exit` when done.